## Data Processing

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import datetime as dt
import matplotlib.pyplot as plt
import yfinance as yf
import torch.nn as nn
import torch

In [3]:
# asset price loading
# dropping Null vals
# , 'TLT', 'XLF', 'XLE', 'XLU', 'XLK', 'EXW1.DE', '^FTSE','1329.T', 'QQQ'
symbols = yf.download(['SPY'],start=dt.datetime(2009,1,1), end=dt.datetime(2019,12,31), auto_adjust=False, multi_level_index=False, progress=False).dropna()

closePs = symbols['Adj Close'].to_frame()

type(closePs)
np.shape(closePs)

(2767, 1)

In [4]:
## Indicators (will be used later)
from ta import volatility
from ta import trend
from ta import momentum
from ta import volume

# Trend
symbols['macd'] = trend.macd(close=symbols['Adj Close'],window_fast=12,window_slow=26)
symbols['ema'] = trend.ema_indicator(close=symbols['Adj Close'],window=14)
symbols['sma'] = trend.sma_indicator(close=symbols['Adj Close'],window=14)
symbols['wma'] = trend.wma_indicator(close=symbols['Adj Close'],window=14)

# Volatility
symbols['atr'] = volatility.average_true_range(high=symbols['High'] ,low=symbols['Low'], close=symbols['Adj Close'],window=14)

# Momentum
symbols['rsi'] = momentum.rsi(close=symbols['Adj Close'],window=14)

# Volume
symbols['mfi'] = volume.money_flow_index(high=symbols['High'] ,low=symbols['Low'],close=symbols['Adj Close'],volume=symbols['Volume'],window=14)

# Self explanatory I believe
symbols["close_vs_sma"] = (symbols["Adj Close"] - symbols["sma"]) / symbols["sma"]

symbols["daily_return"] = symbols["Adj Close"].pct_change()


symbols.dropna()
symbols


,Adj Close,Close,High,Low,Open,Volume,macd,ema,sma,wma,atr,rsi,mfi,close_vs_sma,daily_return
Date,,,,,,,,,,,,,,,
2009-01-02,67.806602,92.959999,93.440002,89.849998,90.440002,227566300,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
2009-01-05,67.726364,92.849998,93.660004,91.889999,92.629997,240349700,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,-0.001183
2009-01-06,68.178581,93.470001,94.449997,92.680000,93.639999,328260900,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0.006677
2009-01-07,66.136238,90.669998,92.260002,90.199997,92.000000,280899200,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,-0.029956
2009-01-08,66.406120,91.040001,91.089996,89.669998,90.160004,263834400,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0.004081
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-23,292.798187,321.220001,321.649994,321.059998,321.589996,52990000,3.208916,288.422305,287.621931,289.351546,30.314000,77.214925,90.751792,0.017997,0.001527
2019-12-24,292.807281,321.230011,321.519989,320.899994,321.470001,20270000,3.278935,289.006968,288.357245,290.042926,30.200272,77.230452,88.208020,0.015432,0.000031
2019-12-26,294.365997,322.940002,322.950012,321.640015,321.649994,30911200,3.420767,289.721505,289.167620,290.844093,30.196161,79.774379,88.096457,0.017977,0.005323


In [5]:
# calculate slopeReference (slopeRef) and find the distribution of the labels
# find the first,second seperation points..

arr = []

for r in range(len(closePs) - 45):
    all_y = closePs.values.tolist()
    price45 = all_y[r + 33][0]
    price30 = all_y[r + 29][0]
    dif_ratio = ((price45 - price30) / price30) * 100
    arr.append(dif_ratio)

arr.sort()
length = len(arr)

f_sliding = round(arr.__getitem__(2*int(length/5)),2)
s_sliding =  round(arr.__getitem__(3*int(length/5)),2)

print("len_roc:",length, "f_sliding:",f_sliding,"s_sliding:",s_sliding)

len_roc: 2722 f_sliding: 0.03 s_sliding: 0.69


In [ ]:
#Series normalization to be in range 0-1
def normalize_series(values, col_name):

      v = np.asarray(values, dtype=np.float64)
      if col_name in ("rsi", "mfi"):
          return np.clip(v / 100.0, 0.0, 1.0)

      if col_name in ("close_vs_sma", "daily_return"):
          clip = 0.05
          v = np.clip(v, -clip, clip)
          return (v / clip) * 0.5 + 0.5

      vmin, vmax = np.nanmin(v), np.nanmax(v)
      if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
          return np.zeros_like(v, dtype=np.float32)

      return ((v - vmin) / (vmax - vmin)).astype(np.float32)


#each feature will be drawn as a 224x224 grayscale image
def draw_channel(norm_60, size=224):

      canvas = np.full((size, size), 255, dtype=np.uint8)
      n = len(norm_60)

      for d in range(n):
          x0 = int(d * size / n)
          x1 = int((d + 1) * size / n)
          if x1 <= x0:
              x1 = x0 + 1

          bar_h = int(float(norm_60[d]) * (size - 1))

          for x in range(x0, min(x1, size)):

              for k in range(bar_h):
                  y = size - 1 - k
                  if y >= 0:
                      canvas[y, x] = 0
      return canvas

In [7]:
# Time Series transformation to images for labeling + analysis
# Using @Omer Berat Sezer's approach
FEATURE_COLS = ["Adj Close", "Volume", "atr", "sma",'macd','ema', "rsi", 'mfi', 'close_vs_sma', 'daily_return']
WINDOW = 60 # trading days per sample (time axis)
IMG_SIZE = 224 # height and width of each channel
FWD_DAYS = 15
df = symbols.dropna().copy()

In [ ]:
def imagesFileCreation(input_file, output_file):

    for r in range(len(input_file) - 45):

        channels = []
        for col in FEATURE_COLS:
            values = window[col].values          # 60 numbers
            normalized = normalize(values, col)  # rule depends on column type
            channel_img = draw_bars(normalized)  # 224×224 grayscale
            channels.append(channel_img)

        #print('r:', r)
        img = [[]]

        # all pixels are set value 255 (white) in 224x224 pixel image
        img = [[255 for i in range(224)] for i in range(224)]
        all_y = input_file.values.tolist()
        sub_y = all_y[r:r + 224]

        current_price = round(all_y[r+29][0], 2)

        price45 = all_y[r + 44][0]
        price30 = all_y[r + 29][0]

        # calculate ratio
        dif_ratio = ((price45 - price30) / price30) * 100

        max_y = (all_y[r+15][0] * 130) / 100
        min_y = (all_y[r+15][0] * 70) / 100

        label_avg_y = 0
        label_sum_y = 0

        #print("f_sliding,s_sliding:",f_sliding,s_sliding)
        #test1
        if (dif_ratio >= s_sliding): # slopeCurrent>slopeRef[s_sliding]
            predictLabel = 1   # label=Buy
        elif (dif_ratio > f_sliding and dif_ratio < s_sliding):
            predictLabel = 0   # label = Hold
        elif (dif_ratio <= f_sliding): # slopeCurrent<slopeRef[s_sliding]
            predictLabel = 2   # label = Sell

        #print("predictLabel:", predictLabel, " dif_ratio:", dif_ratio, "price:", current_price)
        #print("max:",max_y, "min:",min_y)

        # calculate coefficient to normalize data
        coef = 224 / (int(max_y - min_y)+1)
        j = 0
        #print(max_y, min_y)

        # calculate the stock price and create black bar graphics for 30 days
        for i in range(30):
            val = (sub_y[i][0] - min_y) * coef
            #print(val)
            for k in range(int(val)):
                if(k<224):
                    img[223 - k][j] = 0
            j += 1

        my_df = pd.DataFrame(img)
        label_price = ';'.join((str(predictLabel), str(current_price)))

        # append image values in file
        my_df.to_csv(output_file, index=False, header=False, mode='a', lineterminator=';', sep=',')
        with open(output_file, 'a') as file:
            file.write(label_price)
            file.write('\n')
        del label_sum_y, predictLabel

        # print(f'saved at {output_file}') # saving

        # print image
        # print(img)
        # DONT RUN WITH THESE ON EVER
        # imgplot = plt.imshow(img, cmap=plt.get_cmap('gray'))
        # plt.show()

In [22]:
from sklearn.model_selection import train_test_split
#  #Expanding window train-validate splits
# for date in pd.date_range("2009-1-1",'2010-12-31',freq='ME'):
#     training = closePs.loc[:date-pd.offsets.Day(1)]
#     validate = closePs.loc[date:date+pd.offsets.MonthEnd(1)]
#
#     #I am very positive this won't work (yet). I can try to make use of
#     imagesFileCreation(training, 'data/training_file.csv')
#     imagesFileCreation(validate, 'data/test_file.csv')
from multichannel_images import create_multichannel_dataset, load_multichannel_dataset, CHANNEL_NAMES

# 10 channel 224x224 images, 60-day window
n_samples, channel_names = create_multichannel_dataset(symbols,"data/multichannel.npz",f_sliding,s_sliding)

print(n_samples, "samples")
print("channels:", list(channel_names))

2668 samples
channels: ['close', 'volume', 'atr', 'sma', 'ema', 'rsi', 'mfi', 'close_vs_sma', 'daily_return', 'macd']


In [14]:
# validation
#parts = open('data/full224.csv').readline().strip().split(';')

#len(parts)
#len(parts[0].split(','))
#len(parts[:-2])

226

In [23]:
data = load_multichannel_dataset("data/multichannel.npz")

X = data["X"]
y = data["y"]
prices = data["prices"]
channel_names = list(data["channels"])

print("X:", X.shape, "y:", y.shape)
print("channels:", channel_names)

X: (2668, 10, 224, 224) y: (2668,)
channels: [np.str_('close'), np.str_('volume'), np.str_('atr'), np.str_('sma'), np.str_('ema'), np.str_('rsi'), np.str_('mfi'), np.str_('close_vs_sma'), np.str_('daily_return'), np.str_('macd')]


## Data Prep for training


In [6]:
# X is already (n, 10, 224, 224) from multichannel.npz
Y = y
inSplit = int(0.8 * len(Y))
X_train, X_test = X[:inSplit], X[inSplit:]
Y_train, Y_test = Y[:inSplit], Y[inSplit:]

In [7]:
# pixel normalization
X_train = X_train.astype(np.float32) / 255.0
X_test  = X_test.astype(np.float32) / 255.0

In [8]:
print(X_train.shape, X_test.shape)

In [14]:
# Conversion to torch tensors
X_train_t = torch.from_numpy(X_train)
X_test_t = torch.from_numpy(X_test)
Y_train_t = torch.from_numpy(Y_train).long()
Y_test_t = torch.from_numpy(Y_test).long()

## Initial simple training


In [15]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_t, Y_train_t)
test_ds  = TensorDataset(X_test_t, Y_test_t)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True) # shuffle the batches only
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
import torchvision.models as models

n_channels = X_train_t.shape[1]
model = models.resnet18(weights=None)

model.conv1 = nn.Conv2d(n_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Linear(model.fc.in_features, 3)

In [18]:
print(X_train_t.shape, Y_train_t.shape)
print(X_train_t.min(), X_train_t.max())
print(np.bincount(Y_train), np.bincount(Y_test))

torch.Size([2068, 3, 224, 224]) torch.Size([2068])
tensor(0.) tensor(1.)
[ 180 1194  694] [ 47 417 190]


In [19]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

cuda:0


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [24]:
#from tqdm import tqdm
#import torchvision.models as models
#models.list_models()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.01,momentum=0.9,weight_decay=1e-4)

In [22]:
def epoch(model, loader, criterion, optimizer, device):
      model.train()

      running_loss = 0.0
      correct = 0
      total = 0

      for x, y in loader:
          x = x.to(device)
          y = y.to(device)

          optimizer.zero_grad()
          logits = model(x)
          loss = criterion(logits, y)
          loss.backward()
          optimizer.step()

          running_loss += loss.item() * x.size(0)
          preds = logits.argmax(dim=1)
          correct += (preds == y).sum().item()
          total += x.size(0)

      return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
      x = x.to(device)
      y = y.to(device)

      logits = model(x)
      loss = criterion(logits, y)
      running_loss += loss.item() * x.size(0)
      preds = logits.argmax(dim=1)
      correct += (preds == y).sum().item()
      total += x.size(0)

    return running_loss / total, correct / total


In [ ]:
epochs = 50

for e in range(epochs) :

    print ('Epoch {}/{} '.format(e + 1, epochs))

    train_loss, train_acc = epoch(model, train_loader, criterion, optimizer, device)

    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    #for batch in closePs.train:

        # SAM requires 2 forward-backward steps ~

        # # First forward-backward step
        # myOptimizer.firstS()
        # # Second forward-backward step
        # myOptimizer.secondS()

    print(f"train loss {train_loss:.4f}, acc {train_acc:.3f} | "
          f"test loss {test_loss:.4f}, acc {test_acc:.3f}")

In [26]:
torch.save(model.state_dict(), 'data/resnet18_spy.pth')

## Optimizer from scratch


In [ ]:
# Sharpness aware minimization based optim, it's better than SDG w/ momentum
# Need to determine build

class myOptimizer():

    def __init__(self, learning_rate, momentum):

        self.learning_rate = learning_rate
        self.momentum = momentum

        self.w = 0
        self.b = 0

        self.momentum_vector_w = 0
        self.momentum_vector_b = 0


    def _get_batch(self):

    def _get_momentum_vector(self, X_batch, y_batch):

    def firstS(self):

    def secondS(self):

    def fit(self, X, y, batch_size = 32, epochs = 100):

    def predict(self, X):
	    return self.w * X + self.b

In [8]:
#Code copied from personal repro
from src.sam import SAM

def train_epoch_sam(
    model,
    loader,
    criterion,
    optimizer: SAM,
    device,
    use_amp: bool,
):

    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in tqdm(loader, desc="train", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with amp.autocast("cuda"):
                logits = model(x)
                loss = criterion(logits, y)
            loss.backward()
            optimizer.first_step(zero_grad=True)

            with amp.autocast("cuda"):
                logits2 = model(x)
                loss2 = criterion(logits2, y)
            loss2.backward()
            optimizer.second_step(zero_grad=True)
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.first_step(zero_grad=True)

            logits2 = model(x)
            loss2 = criterion(logits2, y)
            loss2.backward()
            optimizer.second_step(zero_grad=True)

        total_loss += loss.detach().float().item() * y.size(0)
        correct += (logits2.argmax(dim=1) == y).sum().item()
        total += y.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)

ModuleNotFoundError: No module named 'src'